# Phase 2 – Preprocessing: Multi-Channel CLAHE

This notebook demonstrates the **CLAHEPreprocessor** pipeline that normalises
OMR sheet images before bubble classification.

## Why CLAHE?

Mobile phone photos of OMR sheets suffer from:
- Uneven illumination (shadows, vignetting)
- JPEG compression block artefacts
- Variable white-balance colour casts

**CLAHE** (Contrast Limited Adaptive Histogram Equalization) corrects local
contrast without amplifying noise, applied to the **L channel in LAB colour
space** to avoid hue distortion.

> Full mathematical derivation in `Phase_2_Preprocessing/LEGACY_REFERENCE.md`.

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add Phase 2 src to the Python path.
phase2_src = Path("../Phase_2_Preprocessing/src").resolve()
if str(phase2_src) not in sys.path:
    sys.path.insert(0, str(phase2_src))

print(f"Phase 2 src path: {phase2_src}")
print(f"Path exists: {phase2_src.exists()}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

from clahe_preprocessor import CLAHEPreprocessor

print(f"CLAHEPreprocessor: {CLAHEPreprocessor}")

## 2. Instantiate the Preprocessor

Key parameters:

| Parameter | Default | Effect |
|---|---|---|
| `clip_limit` | 2.0 | Max histogram bin height before clipping; higher = stronger enhancement but more noise |
| `tile_grid_size` | (8, 8) | Grid of tiles for local histogram computation |
| `denoise` | True | Apply 3×3 Gaussian blur before CLAHE to suppress JPEG artefacts |

In [ ]:
preprocessor = CLAHEPreprocessor(
    clip_limit=2.0,
    tile_grid_size=(8, 8),
    denoise=True,
)
print(f"clip_limit     : {preprocessor.clip_limit}")
print(f"tile_grid_size : {preprocessor.tile_grid_size}")
print(f"denoise        : {preprocessor.denoise}")

## 3. Simulate an Unevenly Lit OMR Form

We create a synthetic test image with a gradient shadow to demonstrate how
CLAHE normalises uneven illumination.

In [ ]:
H, W = 500, 400
base = np.ones((H, W, 3), dtype=np.uint8) * 220

# Add a horizontal gradient (simulates shadow across the form).
gradient = np.linspace(0, 100, W, dtype=np.uint8)
shadow = np.tile(gradient, (H, 1))
base[:, :, 0] = np.clip(base[:, :, 0].astype(int) - shadow, 0, 255).astype(np.uint8)
base[:, :, 1] = np.clip(base[:, :, 1].astype(int) - shadow, 0, 255).astype(np.uint8)
base[:, :, 2] = np.clip(base[:, :, 2].astype(int) - shadow, 0, 255).astype(np.uint8)

# Draw simulated filled and empty bubbles.
for row in range(5):
    for col in range(5):
        cx, cy = 50 + col * 70, 60 + row * 80
        filled = (row + col) % 3 == 0
        colour = (40, 40, 40) if filled else (200, 200, 200)
        cv2.circle(base, (cx, cy), 20, colour, -1)
        cv2.circle(base, (cx, cy), 20, (100, 100, 100), 2)

simulated_form = base.copy()
print(f"Simulated form shape: {simulated_form.shape}")

plt.figure(figsize=(4, 5))
plt.imshow(cv2.cvtColor(simulated_form, cv2.COLOR_BGR2RGB))
plt.title("Simulated OMR Form with Shadow Gradient")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Apply Multi-Channel CLAHE

`process()` pipeline:
1. Optional Gaussian blur (σ=1.0) → attenuate JPEG artefacts
2. BGR → LAB colour space
3. CLAHE on **L channel only** (preserves hue/saturation)
4. LAB → BGR

In [ ]:
processed = preprocessor.process(simulated_form)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(simulated_form, cv2.COLOR_BGR2RGB))
axes[0].set_title("Before CLAHE\n(uneven illumination)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
axes[1].set_title("After CLAHE\n(normalised luminance)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"Input  dtype/shape: {simulated_form.dtype} {simulated_form.shape}")
print(f"Output dtype/shape: {processed.dtype} {processed.shape}")

## 5. Batch Processing

`process_batch()` applies `process()` to every image in a list.

In [ ]:
# Create a small batch of slightly varied copies.
batch = [simulated_form, np.fliplr(simulated_form), np.flipud(simulated_form)]
processed_batch = preprocessor.process_batch(batch)

print(f"Batch size       : {len(processed_batch)}")
print(f"Each output shape: {processed_batch[0].shape}")

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (orig, proc) in enumerate(zip(batch, processed_batch)):
    axes[0, i].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f"Original [{i}]")
    axes[0, i].axis("off")
    axes[1, i].imshow(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f"CLAHE [{i}]")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 6. Convert to PyTorch Tensor

`to_tensor()` converts a preprocessed BGR image to a normalised float32 tensor:

```
BGR (H×W×3, uint8)  →  RGB  →  CHW (3×H×W, float32, values in [0,1])
```

This format is required by `BubbleClassifier` in Phase 3.

In [ ]:
tensor = preprocessor.to_tensor(processed)

print(f"Tensor shape : {tensor.shape}    (C×H×W)")
print(f"Tensor dtype : {tensor.dtype}")
print(f"Value range  : [{tensor.min():.4f}, {tensor.max():.4f}]")
print(f"\nReady to feed into BubbleClassifier (Phase 3).")

## Summary

| Step | Operation | Purpose |
|---|---|---|
| 1 | Gaussian blur (3×3, σ=1.0) | Suppress JPEG block artefacts |
| 2 | BGR → LAB | Separate luminance from colour |
| 3 | CLAHE on L channel | Local contrast normalisation |
| 4 | LAB → BGR | Restore colour |
| 5 | `to_tensor()` | CHW float32 [0,1] for PyTorch |

The preprocessed tensor is passed to **Phase 3 – Classification** for bubble scoring.